In [44]:
import pandas as pd

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
def read_outputs(file_path):
    outputs = pd.read_json(file_path, lines=True)
    outputs = outputs.explode('generations',ignore_index=True)
    outputs['generations'] = outputs['generations'].apply(lambda x: [x])
    return outputs

def unravel(df):
    df['prompt'] = df['prompt'].apply(lambda x: x['text'])
    df['generations'] = df['generations'].apply(lambda x: x[0]['text'])
    return df

In [21]:
## 원본
original = read_outputs('/data/hyeryung/mucoco/new_module/data/sentiment/dev_set_below_positive_threshold_778.jsonl')
original = unravel(original)

## 원본에 locate된 것
locate = read_outputs('/data/hyeryung/mucoco/saeheeeom/new_module/iter_loc_edit_qwen/located/16_pos_located_37837.jsonl_filtered_0')
locate = unravel(locate)

In [33]:
## LLM w/o 랑 w/ 랑 잘되고 안된 샘플 비교하기
llm_w = read_outputs('/data/hyeryung/mucoco/saeheeeom/new_module/iter_loc_edit_qwen/edited/16_pos_edited_38144.jsonl_total_0') 
llm_wo = read_outputs('/data/hyeryung/mucoco/saeheeeom/new_module/iter_loc_edit_qwen/final/18_pos_loc_edit_38398.jsonl')
llm_w = unravel(llm_w)
llm_wo = unravel(llm_wo)
llm_w_senti = pd.read_json('/data/hyeryung/mucoco/saeheeeom/new_module/iter_loc_edit_qwen/edited/16_pos_edited_38144.jsonl_total_0-results.txt.sentiment_ext', lines=True)
llm_wo_senti = pd.read_json('/data/hyeryung/mucoco/saeheeeom/new_module/iter_loc_edit_qwen/final/18_pos_loc_edit_38398.jsonl-results.txt.sentiment_ext', lines=True)

In [34]:
compr_df = pd.DataFrame({
    'prompt': original['prompt'],
    'original': original['generations'],
    'locate': locate['generations'],
    'llm_wo': llm_wo['generations'],
    'llm_w': llm_w['generations'],
    'llm_wo_senti': llm_wo_senti['label'],
    'llm_w_senti': llm_w_senti['label']
})

In [35]:
compr_df['llm_wo_senti'].apply(lambda x: 1 if x == 'POSITIVE' else 0).mean(),\
compr_df['llm_w_senti'].apply(lambda x: 1 if x == 'POSITIVE' else 0).mean()

(0.8817480719794345, 0.7403598971722365)

In [37]:
compr_df.groupby(['llm_wo_senti', 'llm_w_senti']).size()

llm_wo_senti  llm_w_senti
NEGATIVE      NEGATIVE        70
              POSITIVE        22
POSITIVE      NEGATIVE       132
              POSITIVE       554
dtype: int64

In [38]:
compr_indices =compr_df.loc[(compr_df['llm_wo_senti'] == 'POSITIVE')&(compr_df['llm_w_senti']=='NEGATIVE') ].index.tolist()
print(len(compr_indices))

132


In [52]:
compr_df['len'] = compr_df['original'].str.split().apply(len)
compr_df['llm_wo_len'] = compr_df['llm_wo'].str.split().apply(len)
compr_df['llm_w_len'] = compr_df['llm_w'].str.split().apply(len)

In [50]:
compr_df.loc[compr_indices,'len'].value_counts().sort_index()

len
5      1
6      2
7      5
8      5
9     10
10     7
11     2
12     2
13     6
14     4
15     9
16     6
17     8
18     2
27     1
28     1
33     2
34     2
35     2
36     1
37     4
38     6
39    10
40     3
41    11
42     8
43     6
44     3
45     2
46     1
Name: count, dtype: int64

In [53]:
tmp = compr_df.loc[compr_indices].copy()

In [57]:
tmp.shape

(132, 10)

In [56]:
tmp.loc[tmp['len'] < tmp['llm_wo_len']].shape

(93, 10)

In [62]:
tmp.loc[tmp['len'] == tmp['llm_wo_len']].shape

(9, 10)

In [66]:
# tmp.loc[tmp['len'] > tmp['llm_wo_len']].shape
tmp.loc[tmp['len'] > tmp['llm_wo_len']]

,prompt,original,locate,llm_wo,llm_w,llm_wo_senti,llm_w_senti,len,llm_wo_len,llm_w_len
39,The chicken,in a box and the apple cake recipe: one,in a<mask> and the apple cake<mask><mask><mask>,A delicious apple cake recipe: one,in a box and the apple cake recipe: one,POSITIVE,NEGATIVE,9,6,9
98,The horse,was running.\n\n[The horse is running,was<mask><mask>\n\n[The<mask> is<mask>,The horse is galloping.,was running [The horse is running],POSITIVE,NEGATIVE,6,4,6
112,The lake,level will probably have to stay where it is in,level will probably have to<mask><mask><mask> is in,The lake's level will likely remain steady for now.,level will probably have to remain where it is in,POSITIVE,NEGATIVE,10,9,10
355,The horse,man was said to be a man who had been arrested several times on child pornography charges.,man was<mask> to be a<mask> who had been<mask><mask> times on child<mask><mask><mask>,"The horse stands as a symbol of strength and grace, often representing freedom and endurance.","man was said to be a man who had made significant contributions to the community多次在儿童色情相关案件中被指控。 \nNote: The original prompt does not fit well with the masked text provided. If we are to maintain a positive tone, the context might need reevaluation. Assuming the intention was to create a positive context around a man, here is a revised version considering the context might be about someone recognized for their contributions rather than the original negative connotation:\n""man was said to be a man who had made significant contributions to the community""",POSITIVE,NEGATIVE,17,15,90
360,The horse,"that was shot had already died. The horse was a black horse like yours, a horse",that was<mask> had<mask><mask><mask> The<mask> was a black<mask> like<mask><mask> a horse,"The black horse, like yours, had calmly passed away.","that was killed had already died. The horse was a black horse like yours, a horse",POSITIVE,NEGATIVE,16,9,16
369,The horse,'s hoof fell out from under it and rolled into the trough. Now the woman could,'s<mask><mask><mask> out from under it and<mask> into the<mask><mask> Now the woman<mask>,"The horse's hoof slipped and rolled into the trough, but now the woman could",'s slipping out from under it and rolling into the trough. Now the woman could,POSITIVE,NEGATIVE,16,14,15
372,The lake,", however, is not a viable option in the short term.\n\n""The lake",",<mask><mask> is not a<mask><mask> in the<mask> term.<mask><mask><mask><mask><mask>",", remains an exciting long-term vision.""","""The lake , is not a feasible option in the short term.""",POSITIVE,NEGATIVE,13,6,12
419,The movie,"has yet to appear on a wide release, but it should appear by the time it does",has<mask> to<mask><mask> a<mask><mask><mask><mask> it should appear by the time it does,"The movie hasn't appeared in wide release yet, but it's expected to do so soon.",has potential to appear it should appear by the time it does,POSITIVE,NEGATIVE,17,15,12
459,The potato,"was an unknown, perhaps unknown variety (not related to the potato), which would have become",was an<mask><mask><mask><mask><mask> (not<mask> to the<mask><mask> which would have become,was a fascinating variety (possibly previously unknown) that had the potential to become,"was an unknown, perhaps a rare variety (not related to the potato), which would have become",POSITIVE,NEGATIVE,15,13,16
466,The potato,"is not going to get out of the soil!""\n\nIt's true that potatoes are",is<mask><mask> to get out of the<mask><mask>\n\nIt's<mask> that<mask> are,"is deeply rooted in the soil, isn't it?It's true that potatoes are",isn't going to get out of the soil!It's true that potatoes are,POSITIVE,NEGATIVE,14,12,12


In [ ]:
tmp.loc[tmp['len'] < tmp['llm_w_len']].shape

(11, 10)

In [61]:
tmp.loc[tmp['len']== tmp['llm_w_len']].shape

(61, 10)

In [65]:
tmp.loc[tmp['len'] > tmp['llm_w_len']].shape

(60, 10)

In [59]:
tmp.loc[tmp['original'] == tmp['llm_w']].shape

(8, 10)

In [60]:
tmp.loc[tmp['original'] == tmp['llm_wo']].shape

(0, 10)

In [ ]:
# L&E랑 LLM EDIT w/랑 비교하기